# Phase 3: Visualizing PDFs & PMFs

Explore how each distribution's shape changes as its parameters vary.
These four distributions directly connect to the loss functions we'll derive next:

| Distribution | MLE → Loss | Task |
|---|---|---|
| Gaussian | MSE | Regression |
| Laplacian | L1 / Huber | Robust regression |
| Bernoulli | BCE | Binary classification |
| Categorical | CrossEntropy | Multi-class classification |

In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parent) if Path.cwd().name == "apps" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.append(project_root)

import matplotlib.pyplot as plt
import numpy as np
from core.prob import Bernoulli, Categorical, Gaussian, Laplacian

---
## 1. Gaussian $\mathcal{N}(\mu, \sigma^2)$

$$p(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

- $\mu$ controls **location** (center of the bell)
- $\sigma$ controls **spread** (width of the bell)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
x = np.linspace(-5, 5, 500)

# Varying mu, fixed sigma
for mu in [-2, 0, 2]:
    g = Gaussian(mu=mu, sigma=1)
    ax1.plot(x, g.pdf(x), label=f"$\\mu={mu}$, $\\sigma=1$")
ax1.set_title("Varying $\\mu$")
ax1.set_xlabel("x")
ax1.set_ylabel("p(x)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Varying sigma, fixed mu
for sigma in [0.5, 1.0, 2.0]:
    g = Gaussian(mu=0, sigma=sigma)
    ax2.plot(x, g.pdf(x), label=f"$\\mu=0$, $\\sigma={sigma}$")
ax2.set_title("Varying $\\sigma$")
ax2.set_xlabel("x")
ax2.set_ylabel("p(x)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()

---
## 2. Laplacian $\text{Lap}(\mu, b)$

$$p(x) = \frac{1}{2b} \exp\left(-\frac{|x-\mu|}{b}\right)$$

- Heavier tails than Gaussian → more tolerant of outliers
- $\mu$ controls location, $b$ controls spread ($\text{Var}=2b^2$)
- Compare Gaussian vs Laplacian with same mean and variance

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Varying mu, fixed b
for mu in [-2, 0, 2]:
    lap = Laplacian(mu=mu, b=1)
    ax1.plot(x, lap.pdf(x), label=f"$\\mu={mu}$, $b=1$")
ax1.set_title("Laplacian: Varying $\\mu$")
ax1.set_xlabel("x")
ax1.set_ylabel("p(x)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Gaussian vs Laplacian: same (mu=0) and same variance
# Var[Lap] = 2b^2 => to match Var=1, set b = 1/sqrt(2)
g = Gaussian(mu=0, sigma=1)
lap = Laplacian(mu=0, b=1 / np.sqrt(2))
ax2.plot(x, g.pdf(x), label="Gaussian: $\\sigma=1$", linestyle="-")
ax2.plot(x, lap.pdf(x), label="Laplacian: $b=1/\\sqrt{2}$", linestyle="--")
ax2.set_title("Gaussian vs Laplacian (same variance)")
ax2.set_xlabel("x")
ax2.set_ylabel("p(x)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()

---
## 3. Bernoulli $\text{Bern}(p)$

$$p(x) = p^x (1-p)^{1-x}, \quad x \in \{0, 1\}$$

- $p$ controls the probability of $x=1$ ("success")
- $\mathbb{E}[X] = p$, $\text{Var}[X] = p(1-p)$ — variance is maximized at $p=0.5$

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# PMF bars for different p values
categories = [0, 1]
for p, color in zip([0.2, 0.5, 0.8], ["steelblue", "green", "coral"]):
    b = Bernoulli(p=p)
    probs = [b.pmf(0), b.pmf(1)]
    ax1.bar(
        [c - 0.2 + 0.2 * p / 0.3 for c in categories],
        probs,
        width=0.15,
        color=color,
        alpha=0.7,
        label=f"p={p}",
    )
ax1.set_xticks([0, 1])
ax1.set_title("Bernoulli PMF")
ax1.set_ylabel("P(X=x)")
ax1.legend()
ax1.grid(True, alpha=0.3, axis="y")

# Variance vs p curve
ps = np.linspace(0, 1, 100)
vars = ps * (1 - ps)
ax2.plot(ps, vars)
ax2.axvline(0.5, color="gray", linestyle="--", alpha=0.5)
ax2.set_title("Bernoulli Variance $\\text{Var}[X] = p(1-p)$")
ax2.set_xlabel("p")
ax2.set_ylabel("Variance")
ax2.grid(True, alpha=0.3)

plt.tight_layout()

---
## 4. Categorical $\text{Cat}(\mathbf{p})$

$$p(x=k) = p_k, \quad \sum_{k=1}^K p_k = 1, \quad p_k \ge 0$$

- Generalizes Bernoulli to $K$ categories
- $\mathbb{E}[\text{one-hot}(X)] = \mathbf{p}$, $\text{Cov} = \text{diag}(\mathbf{p}) - \mathbf{p}\mathbf{p}^T$

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Two different probability vectors
configs = [
    ("Uniform", np.array([0.25, 0.25, 0.25, 0.25])),
    ("Skewed", np.array([0.5, 0.3, 0.15, 0.05])),
]

x_pos = np.arange(4)
width = 0.3
for i, (label, probs) in enumerate(configs):
    ax1.bar(
        x_pos + i * width - width / 2,
        probs,
        width=width,
        alpha=0.7,
        label=label,
    )
ax1.set_xticks(x_pos)
ax1.set_title("Categorical PMF (K=4)")
ax1.set_xlabel("Category k")
ax1.set_ylabel("P(X=k)")
ax1.legend()
ax1.grid(True, alpha=0.3, axis="y")

# Sampling demonstration: draw many samples and compare with theoretical probs
np.random.seed(42)
probs = np.array([0.1, 0.2, 0.4, 0.3])
c = Categorical(probs=probs)
samples = c.sample(10000)
empirical = np.bincount(samples, minlength=4) / 10000

ax2.bar(
    x_pos - 0.15, probs, width=0.3, alpha=0.7, label="Theoretical", color="steelblue"
)
ax2.bar(
    x_pos + 0.15,
    empirical,
    width=0.3,
    alpha=0.7,
    label="Empirical (10k samples)",
    color="coral",
)
ax2.set_xticks(x_pos)
ax2.set_title("Sampling: Theoretical vs Empirical")
ax2.set_xlabel("Category k")
ax2.set_ylabel("Probability")
ax2.legend()
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()

---
## Summary

### Shape comparison

| Distribution | Support | Key Parameter(s) | Shape | Tail |
|---|---|---|---|---|
| Gaussian | $(-\infty, \infty)$ | $\mu$, $\sigma$ | Bell curve | Thin ($e^{-x^2}$) |
| Laplacian | $(-\infty, \infty)$ | $\mu$, $b$ | Sharp peak | Heavy ($e^{-\|x\|}$) |
| Bernoulli | $\{0, 1\}$ | $p$ | Two bars | — |
| Categorical | $\{1, ..., K\}$ | $\mathbf{p}$ | Multiple bars | — |